In [1]:
"""
Lu Section 4 simulations
"""

import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import time
import pickle

from dataclasses import dataclass, asdict

In [2]:
project_root = os.path.abspath(os.getcwd())
sys.path.insert(0, project_root)


from BayesianSparseDeepHalo.LuSparseRandomLogit import BayesianSparseRandomLogit, calibrate_stepsizes_cl
from BayesianSparseDeepHalo.GenerateData import generate_data_tf
from BayesianSparseDeepHalo.BLP import blp_estimator_fast

In [3]:
# =====================================
# PARAMETERS
# =====================================

# Specify beta update method: "tmh" or "rwmh"
beta_method = "tmh"

@dataclass
class SimParams:
    """Data Generation Parameters"""
    T: int = 100  # Number of Markets
    J: int = 15   # Number of Products per Market
    Nt: int = 1000  # Consumers per Market
    D: int = 2    # Number of features
    beta_mean: np.ndarray = None  # Mean of beta (D,)
    sigma_beta: np.ndarray = None  # Std of beta (D,)
    xi_bar: float = -1.0
    seed: int = 123

    def __post_init__(self):
        if self.beta_mean is None:
            self.beta_mean = np.array([-1.0, 0.5])
        if self.sigma_beta is None:
            self.sigma_beta = np.array([1.5, 0.0])  # Price random, w fixed (0 means fixed)

@dataclass
class MCMCParams:
    """Hyperparameters for the Bayesian Algorithm"""
    R0: int = 200  # Fixed simulation draws
    G: int = 2000  # Total iterations
    burn: int = 500  # Burn-in samples

    # Masking: which beta components are random (1) vs fixed (0)
    random_coef_mask: np.ndarray = None

    # Priors
    tau0: float = 1e-3
    tau1: float = 1.0
    a_phi: float = 1.0
    b_phi: float = 1.0
    V_beta: float = 10.0
    V_xibar: float = 10.0
    V_r: float = 0.5

    # Initial Steps
    kappa_beta: float = 2.38
    step_beta: float = 0.05
    step_r: float = 0.1
    step_xibar: float = 0.2
    step_eta: float = 1.0

    # Feature dimension
    D: int = 2

    def __post_init__(self):
        if self.random_coef_mask is None:
            # Default: first coefficient is random, rest are fixed
            self.random_coef_mask = np.zeros(self.D)
            self.random_coef_mask[0] = 1.0

@dataclass
class CalibParams:
    """Settings for the Calibration Phase"""
    calib_iters: int = 500
    burn_in: int = 50
    adapt_every: int = 25
    accept_target_low: float = 0.30
    accept_target_high: float = 0.50
    upscale_ratio: float = 1.10
    downscale_ratio: float = 0.90
    min_step: float = 1e-4
    max_step: float = 5.0



In [4]:
# =====================================
# MCMC and BLP RUNNERS
# =====================================

def run_choice_learn_mcmc(choice_dataset, sim_params, mcmc_params, **kwargs):
    model = BayesianSparseRandomLogit(sim_params=sim_params, mcmc_params=mcmc_params, **kwargs)
    losses = model.fit(choice_dataset, store_samples=True)
    return model.samples_, losses, model
    

def _blp_estimate_one(data, true, N_draw=500, seed=111):
    u_cost = true["u_cost"]
    out1 = blp_estimator_fast(data, u_cost, iv_type=1, N_draw=N_draw, seed=seed)
    out2 = blp_estimator_fast(data, u_cost, iv_type=2, N_draw=N_draw, seed=seed)
    return out1, out2


In [5]:
# =====================================
# METRICS & UTILS
# =====================================

def _compute_metrics_one_rep(draws: dict, true: dict, sim: SimParams):
    # Lu2025 posterior-mean estimates
    beta_draws = draws["beta_bar"]          # (keep, D)
    r_draws = draws["r_vec"]                # (keep, n_random)
    xibar_draws = draws["xi_bar"]           # (keep, T)
    eta_draws = draws["eta"]                # (keep, T, J)
    gamma_draws = draws["gamma"]            # (keep, T, J)

    beta_hat = beta_draws.mean(axis=0)
    sigma_hat_vec = np.exp(r_draws).mean(axis=0)
    xibar_hat = xibar_draws.mean(axis=0)

    # xi_{t,j} = xi_bar_t + eta_{t,j}
    xi_draws = xibar_draws[:, :, None] + eta_draws
    xi_hat = xi_draws.mean(axis=0)         # (T, J)

    # posterior inclusion probability
    gamma_prob = gamma_draws.mean(axis=0)  # (T, J)

    # condition on TRUE eta_star being zero vs nonzero
    eta_true = true["eta_star"]
    mask_nz = (np.abs(eta_true) > 1e-12)
    mask_z = ~mask_nz

    # Store generic metrics
    metrics = {
        "xibar_hat_mean": float(xibar_hat.mean()),
        "xi_hat": xi_hat,
        "xi_true": true["xi_star"],
        "pr_gamma_1_eta_nz": float(gamma_prob[mask_nz].mean()) if mask_nz.any() else np.nan,
        "pr_gamma_1_eta_z": float(gamma_prob[mask_z].mean()) if mask_z.any() else np.nan,
    }

    # Add betas
    for d in range(sim.D):
        metrics[f"beta_{d}_hat"] = float(beta_hat[d])

    # Add sigmas (assuming 1st random coef matches sigma_p for D=2 compat)
    if len(sigma_hat_vec) > 0:
        metrics["sigma_hat"] = float(sigma_hat_vec[0])
    else:
        metrics["sigma_hat"] = 0.0

    return metrics

def _summarize_across_reps(rep_metrics: list, sim: SimParams):
    # Aggregation
    xibar_hats = np.array([met["xibar_hat_mean"] for met in rep_metrics])

    out = {
        "bias_xibar": float(xibar_hats.mean() - sim.xi_bar),
        "sd_xibar": float(xibar_hats.std(ddof=1)),
        "pr_gamma_1_eta_nz": float(np.nanmean([met["pr_gamma_1_eta_nz"] for met in rep_metrics])),
        "pr_gamma_1_eta_z": float(np.nanmean([met["pr_gamma_1_eta_z"] for met in rep_metrics])),
    }

    # Beta stats
    for d in range(sim.D):
        b_hats = np.array([met[f"beta_{d}_hat"] for met in rep_metrics])
        out[f"bias_beta_{d}"] = float(b_hats.mean() - sim.beta_mean[d])
        out[f"sd_beta_{d}"] = float(b_hats.std(ddof=1))

    # Sigma stats (only 1st one for table compat)
    # Assuming sim.sigma_beta[0] is the target
    s_hats = np.array([met["sigma_hat"] for met in rep_metrics])
    target_sigma = sim.sigma_beta[0] if sim.sigma_beta is not None else 0.0
    out["bias_sigma"] = float(s_hats.mean() - target_sigma)
    out["sd_sigma"] = float(s_hats.std(ddof=1))

    # Xi bias
    xi_hat_stack = np.stack([met["xi_hat"] for met in rep_metrics], axis=0)  
    xi_true = rep_metrics[0]["xi_true"]
    bias_tj = xi_hat_stack.mean(axis=0) - xi_true
    sd_tj = xi_hat_stack.std(axis=0, ddof=1)
    out["avg_abs_bias_xi_jt"] = float(np.abs(bias_tj).mean())
    out["avg_sd_xi_jt"] = float(sd_tj.mean())

    return out



def _compute_blp_metrics_one_rep(blp_out: dict, true: dict, sim: SimParams):
    """
    blp_out: output dict from blp_estimator_fast 
             must contain keys: "theta", "xi", "xi_bar"
    true: must contain "xi_star" (T,J)
    sim: provides beta_mean, sigma_beta, xi_bar (scalar truth)

    Returns a dict 
    """
    theta = np.asarray(blp_out["theta"]).reshape(-1)  # (3,) [beta_p, beta_w, sigma_p]
    xi_hat = np.asarray(blp_out["xi"])               # (T,J) inside only
    xibar_hat_t = np.asarray(blp_out["xi_bar"]).reshape(-1)  # (T,)

    met = {
        # parameter point estimates
        "beta_0_hat": float(theta[0]),
        "beta_1_hat": float(theta[1]),
        "sigma_hat": float(theta[2]),

        # xibar: store market-average estimate (scalar) for bias/sd 
        "xibar_hat_mean": float(xibar_hat_t.mean()),

        # xi objects for Lu-style abs bias/sd over (t,j)
        "xi_hat": xi_hat,
        "xi_true": np.asarray(true["xi_star"]),
    }
    return met


def _summarize_blp_across_reps(rep_metrics: list, sim: SimParams):
    """
    Aggregates BLP rep metrics 
    """
    out = {}

    # xibar (scalar truth)
    xibar_hats = np.array([m["xibar_hat_mean"] for m in rep_metrics], dtype=float)
    out["bias_xibar"] = float(xibar_hats.mean() - float(sim.xi_bar))
    out["sd_xibar"] = float(xibar_hats.std(ddof=1))

    # betas
    b0 = np.array([m["beta_0_hat"] for m in rep_metrics], dtype=float)
    b1 = np.array([m["beta_1_hat"] for m in rep_metrics], dtype=float)
    out["bias_beta_0"] = float(b0.mean() - float(sim.beta_mean[0]))
    out["sd_beta_0"] = float(b0.std(ddof=1))
    out["bias_beta_1"] = float(b1.mean() - float(sim.beta_mean[1]))
    out["sd_beta_1"] = float(b1.std(ddof=1))

    # sigma (target is first random coef, for D=2 compat)
    s = np.array([m["sigma_hat"] for m in rep_metrics], dtype=float)
    target_sigma = float(sim.sigma_beta[0]) if sim.sigma_beta is not None else 0.0
    out["bias_sigma"] = float(s.mean() - target_sigma)
    out["sd_sigma"] = float(s.std(ddof=1))

    # xi_{t,j} Lu-style abs bias/sd
    xi_hat_stack = np.stack([m["xi_hat"] for m in rep_metrics], axis=0)   # (n_rep,T,J)
    xi_true_stack = np.stack([m["xi_true"] for m in rep_metrics], axis=0) # (n_rep,T,J)

    bias_tj = xi_hat_stack.mean(axis=0) - xi_true_stack.mean(axis=0)      # (T,J)
    sd_tj = xi_hat_stack.std(axis=0, ddof=1)                               # (T,J)
    out["avg_abs_bias_xi_jt"] = float(np.abs(bias_tj).mean())
    out["avg_sd_xi_jt"] = float(sd_tj.mean())

    return out


    
    
# =====================================
# PLOTTING & MAIN RUNNER
# =====================================
def plot_mcmc_and_probs(draws, true, sim, blp_iv1=None, blp_iv2=None,
                        blp1_xibar=None, blp2_xibar=None, out_path_prefix=None):
    beta_draws = draws["beta_bar"]                 # (N, D)
    sigma_draws = np.exp(draws["r_vec"])           # (N, n_random)
    phi_mean = draws["phi"].mean(axis=0)           # (T,)


    xibar_draws = draws["xi_bar"]                  # (N, T)
    xibar_trace = xibar_draws.mean(axis=1)         # (N,)

    D = sim.D
    n_random = sigma_draws.shape[1]


    n_rows = D + n_random + 2
    fig, axes = plt.subplots(n_rows, 2, figsize=(14, 4*n_rows))

    # Plot Betas 
    for d in range(D):
        b_trace = beta_draws[:, d]
        true_b = sim.beta_mean[d]

        ax = axes[d, 0]
        ax.plot(b_trace, alpha=0.7)
        ax.axhline(true_b, color='r', ls='--', label='True')
        ax.axhline(b_trace.mean(), color='k', ls='-', label='Posterior Mean')
        if blp_iv1 is not None: ax.axhline(blp_iv1[d], color='b', ls=':', label='BLP1')
        if blp_iv2 is not None: ax.axhline(blp_iv2[d], color='purple', ls=':', label='BLP2')
        ax.set_title(f"Beta {d} Samples")
        ax.legend()

        ax = axes[d, 1]
        ax.hist(b_trace, bins=30, alpha=0.6)
        ax.axvline(true_b, color='r', ls='--', label='True')
        ax.axvline(b_trace.mean(), color='k', ls='-', label='Posterior Mean')
        if blp_iv1 is not None: ax.axvline(blp_iv1[d], color='b', ls=':', label='BLP1')
        if blp_iv2 is not None: ax.axvline(blp_iv2[d], color='purple', ls=':', label='BLP2')
        ax.set_title(f"Beta {d} Posterior")
        ax.legend()

    # Plot Sigmas
    offset = D
    for k in range(n_random):
        s_trace = sigma_draws[:, k]
        true_s = sim.sigma_beta[sim.sigma_beta > 1e-6][k] if k < np.sum(sim.sigma_beta > 1e-6) else 0

        row = D + k
        ax = axes[row, 0]
        ax.plot(s_trace, alpha=0.7)
        ax.axhline(s_trace.mean(), color='k', ls='-', label='Posterior Mean')
        ax.axhline(true_s, color='r', ls='--', label='True')
        if blp_iv1 is not None: ax.axhline(blp_iv1[offset+k], color='b', ls=':', label='BLP1')
        if blp_iv2 is not None: ax.axhline(blp_iv2[offset+k], color='purple', ls=':', label='BLP2')
        ax.set_title(f"Sigma {k} Samples")
        ax.legend()

        ax = axes[row, 1]
        ax.hist(s_trace, bins=30, alpha=0.6)
        ax.axvline(s_trace.mean(), color='k', ls='-', label='Posterior Mean')
        ax.axvline(true_s, color='r', ls='--', label='True')
        if blp_iv1 is not None: ax.axvline(blp_iv1[offset+k], color='b', ls=':', label='BLP1')
        if blp_iv2 is not None: ax.axvline(blp_iv2[offset+k], color='purple', ls=':', label='BLP2')
        ax.set_title(f"Sigma {k} Posterior")
        ax.legend()

    # Plot xi_bar (scalar per draw)
    xibar_row = D + n_random
    true_xibar = float(sim.xi_bar)
    blp1_xibar_mean = float(np.mean(blp1_xibar)) if blp1_xibar is not None else None
    blp2_xibar_mean = float(np.mean(blp2_xibar)) if blp2_xibar is not None else None

    ax = axes[xibar_row, 0]
    ax.plot(xibar_trace, alpha=0.7)
    ax.axhline(true_xibar, color='r', ls='--', label='True')
    ax.axhline(xibar_trace.mean(), color='k', ls='-', label='Posterior Mean')
    if blp1_xibar_mean is not None: ax.axhline(blp1_xibar_mean, color='b', ls=':', label='BLP1')
    if blp2_xibar_mean is not None: ax.axhline(blp2_xibar_mean, color='purple', ls=':', label='BLP2')
    ax.set_title("xi_bar (mean over t) Samples")
    ax.legend()

    ax = axes[xibar_row, 1]
    ax.hist(xibar_trace, bins=30, alpha=0.6)
    ax.axvline(true_xibar, color='r', ls='--', label='True')
    ax.axvline(xibar_trace.mean(), color='k', ls='-', label='Posterior Mean')
    if blp1_xibar_mean is not None: ax.axvline(blp1_xibar_mean, color='b', ls=':', label='BLP1')
    if blp2_xibar_mean is not None: ax.axvline(blp2_xibar_mean, color='purple', ls=':', label='BLP2')
    ax.set_title("xi_bar (mean over t) Posterior")
    ax.legend()

    # Plot Phi / Gamma (row shifts down by 1)
    last_row = n_rows - 1
    axes[last_row, 0].plot(phi_mean)
    axes[last_row, 0].set_title("Phi Mean (Market Inclusion)")

    gamma_prob = draws["gamma"].mean(axis=0).ravel()
    axes[last_row, 1].hist(gamma_prob, bins=30, range=(0, 1))
    axes[last_row, 1].set_title("Gamma Inclusion Probs")

    plt.tight_layout()
    if out_path_prefix:
        plt.savefig(out_path_prefix + "_mcmc.png")
        plt.close(fig)
    else:
        plt.show()


# =====================================
# MAIN RUNNER
# =====================================

def run_global_grid(
    n_rep=10,
    Ts=(25, 100),
    Js=(5, 15),
    DGPs=(1, 2),
    Nt=1000,
    base_seed=123,
    out_dir=os.path.join(project_root, 'Experiments', 'Lu DGP results '+beta_method ),
    N_draw=300,
    blp_seed=111,
    mcmc_template=None,
    calib_template=None,
    D=2,
    save_pickles=True, 
    make_plots=True, 
    plot_every_rep=False
):
    os.makedirs(out_dir, exist_ok=True)

    # Default Config for D=2 (Original Replication)
    if D == 2:
        beta_mean = np.array([-1.0, 0.5])
        sigma_beta = np.array([1.5, 0.0])
        random_mask = np.array([1.0, 0.0])
    else:
        beta_mean = np.random.randn(D)
        sigma_beta = np.zeros(D); sigma_beta[0]=1.0
        random_mask = np.zeros(D); random_mask[0]=1.0

    print(f"True beta: {beta_mean}")
    print(f"True sigma (std): {sigma_beta[0]}")

    if mcmc_template is None:
        mcmc_template = MCMCParams(R0=200, G=2000, burn=400, D=D, random_coef_mask=random_mask)
    if calib_template is None:
        calib_template = CalibParams(calib_iters=500, burn_in=100)

    for dgp in DGPs:
        rows = []
        for J in Js:
            for T in Ts:
                print(f"Processing DGP={dgp}, T={T}, J={J}...")
                blp1_rep, blp2_rep, lu_rep, rep_out = [], [], [], []


                for rep in range(n_rep):
                    seed = base_seed + 100000 * dgp + 1000 * T + 10 * J + rep
                    sim = SimParams(T=T, J=J, Nt=Nt, D=D, beta_mean=beta_mean, sigma_beta=sigma_beta, seed=seed)
                    data, dataset_cl, true = generate_data_tf(dgp, sim)

                
                    # --- BLP
                    blp_out1, blp_out2 = _blp_estimate_one(data, true, N_draw=N_draw, seed=blp_seed + seed)
                    
                    th_iv1 = np.asarray(blp_out1["theta"]).reshape(-1)
                    th_iv2 = np.asarray(blp_out2["theta"]).reshape(-1)
                    
                    blp1_rep.append(_compute_blp_metrics_one_rep(blp_out1, true, sim))
                    blp2_rep.append(_compute_blp_metrics_one_rep(blp_out2, true, sim))


                    # --- Lu2025
                    mcmc = MCMCParams(**vars(mcmc_template))
                    calib = CalibParams(**vars(calib_template))
                    mcmc = calibrate_stepsizes_cl(dataset_cl, sim, mcmc, calib, beta_method=beta_method, verbose=True,)
                    draws, _, _ = run_choice_learn_mcmc(dataset_cl, sim, mcmc, beta_method=beta_method, verbose=True,)
                    




                    metric = _compute_metrics_one_rep(draws, true, sim)
                    lu_rep.append(metric)

                    # Store Rep Result
                    res_dict = {"rep": rep, "DGP": dgp, "T": T, "J": J}
                    # Flatten metrics
                    for k, v in metric.items():
                        if isinstance(v, (float, int)): res_dict["lu_"+k] = v             

                    for k_idx, name in enumerate(["beta_0", "beta_1", "sigma"][:len(th_iv1)]):
                        res_dict[f"blp1_{name}"] = float(th_iv1[k_idx])
                        res_dict[f"blp2_{name}"] = float(th_iv2[k_idx])

                    res_dict["blp1_xibar_hat_mean"] = blp1_rep[-1]["xibar_hat_mean"]
                    res_dict["blp2_xibar_hat_mean"] = blp2_rep[-1]["xibar_hat_mean"]


                    rep_out.append(res_dict)

                    # Save Pickles
                    if save_pickles:
                        pkl_path = os.path.join(out_dir, f"full_DGP{dgp}_T{T}_J{J}_rep{rep}.pkl")
                        with open(pkl_path, "wb") as f:
                            pickle.dump({"draws": draws, "blp1": blp_out1, "blp2": blp_out2}, f)

                    # Plot
                    if make_plots and (plot_every_rep or rep == 0):
                        prefix = os.path.join(out_dir, f"plot_DGP{dgp}_T{T}_J{J}_rep{rep}")
                        plot_mcmc_and_probs(
                            draws, true, sim,
                            blp_iv1=th_iv1, blp_iv2=th_iv2,
                            blp1_xibar=np.asarray(blp_out1["xi_bar"]).reshape(-1),
                            blp2_xibar=np.asarray(blp_out2["xi_bar"]).reshape(-1),
                            out_path_prefix=prefix
                        )

                # --- Summarize
                # Lu summary
                lu_summary = _summarize_across_reps(lu_rep, sim)
                blp1_summary = _summarize_blp_across_reps(blp1_rep, sim)
                blp2_summary = _summarize_blp_across_reps(blp2_rep, sim)



                row = {
                    "DGP": dgp, "T": T, "J": J,
                
                    # Lu
                    "lu_bias_xibar": lu_summary["bias_xibar"],
                    "lu_sd_xibar": lu_summary["sd_xibar"],
                    "lu_bias_beta_0": lu_summary["bias_beta_0"],
                    "lu_sd_beta_0": lu_summary["sd_beta_0"],
                    "lu_bias_beta_1": lu_summary["bias_beta_1"],
                    "lu_sd_beta_1": lu_summary["sd_beta_1"],
                    "lu_bias_sigma_beta_0": lu_summary["bias_sigma"],
                    "lu_sd_sigma_beta_0": lu_summary["sd_sigma"],
                    "lu_avg_abs_bias_xi_jt": lu_summary["avg_abs_bias_xi_jt"],
                    "lu_avg_sd_xi_jt": lu_summary["avg_sd_xi_jt"],
                
                    # BLP1
                    "blp1_bias_xibar": blp1_summary["bias_xibar"],
                    "blp1_sd_xibar": blp1_summary["sd_xibar"],
                    "blp1_bias_beta_0": blp1_summary["bias_beta_0"],
                    "blp1_sd_beta_0": blp1_summary["sd_beta_0"],
                    "blp1_bias_beta_1": blp1_summary["bias_beta_1"],
                    "blp1_sd_beta_1": blp1_summary["sd_beta_1"],
                    "blp1_bias_sigma_beta_0": blp1_summary["bias_sigma"],
                    "blp1_sd_sigma_beta_0": blp1_summary["sd_sigma"],
                    "blp1_avg_abs_bias_xi_jt": blp1_summary["avg_abs_bias_xi_jt"],
                    "blp1_avg_sd_xi_jt": blp1_summary["avg_sd_xi_jt"],
                
                    # BLP2
                    "blp2_bias_xibar": blp2_summary["bias_xibar"],
                    "blp2_sd_xibar": blp2_summary["sd_xibar"],
                    "blp2_bias_beta_0": blp2_summary["bias_beta_0"],
                    "blp2_sd_beta_0": blp2_summary["sd_beta_0"],
                    "blp2_bias_beta_1": blp2_summary["bias_beta_1"],
                    "blp2_sd_beta_1": blp2_summary["sd_beta_1"],
                    "blp2_bias_sigma_beta_0": blp2_summary["bias_sigma"],
                    "blp2_sd_sigma_beta_0": blp2_summary["sd_sigma"],
                    "blp2_avg_abs_bias_xi_jt": blp2_summary["avg_abs_bias_xi_jt"],
                    "blp2_avg_sd_xi_jt": blp2_summary["avg_sd_xi_jt"],
                }

    
                rows.append(row)

                # Save Reps
                pd.DataFrame(rep_out).to_csv(os.path.join(out_dir, f"reps_DGP{dgp}_T{T}_J{J}.csv"), index=False)

        # Save Global Table
        pd.DataFrame(rows).to_csv(os.path.join(out_dir, f"table_DGP{dgp}.csv"), index=False)
        print(f"Saved table for DGP {dgp}")



In [6]:
run_global_grid(n_rep=5,
        DGPs=(1, 2, 3, 4),
        Ts=(25, 100),
        Js=(5, 15),
        D=2)

True beta: [-1.   0.5]
True sigma (std): 1.5
Processing DGP=1, T=25, J=5...


BLP IV1 stage1: 84it [00:00, 410.16it/s, calls=84, obj=6.2779e+04]
BLP IV1 stage2: 16it [00:00, 103.79it/s, calls=16, obj=4.8080e+05]


BLP 1 final beta: [-1.54131499  0.55809359]
BLP 1 final sigma (std): 0.7496059712337626


BLP IV2 stage1: 84it [00:00, 337.38it/s, calls=84, obj=1.9121e+06]
BLP IV2 stage2: 84it [00:00, 419.51it/s, calls=84, obj=1.2674e+02]


BLP 2 final beta: [-1.18719485  0.59122441]
BLP 2 final sigma (std): 2.8894674366914663
>>> Starting Pilot Calibration...


...

Calibrating: 100%|████████████████████████████████████████████████████████████████████| 500/500 [00:54<00:00,  9.12it/s]


Calibration Done. Tuned: kappa_beta=1.718, step_r=0.031, step_xibar=0.266, step_eta=0.430


Main MCMC chain: 100%|██████████████████████████████████████████████████████████████| 2000/2000 [03:24<00:00,  9.76it/s]

Final beta: [-0.95744277  0.52828187]
Final sigma (std): [1.50741204]
Saved table for DGP 4



/var/folders/z9/wlmv5z6x1h96wq3x7djmx_2w0000gn/T/ipykernel_4286/2305558636.py:58: RuntimeWarning: Mean of empty slice
  "pr_gamma_1_eta_z": float(np.nanmean([met["pr_gamma_1_eta_z"] for met in rep_metrics])),
